# 🚀 NVIDIA Nemotron-3-Nano-30B: First Submission Notebook

Welcome to the **NVIDIA Nemotron Model Reasoning Challenge**! 

This notebook serves as our **first baseline submission pipeline**. Before writing complex training code, it is critical to verify the end-to-end evaluation pipeline on Kaggle. This notebook:
1. Loads the official `NVIDIA Nemotron-3-Nano-30B` baseline model.
2. Configures a compatible LoRA adapter with rank <= 32.
3. Packages the adapter config and weight files into a `submission.zip` file.
4. Explains how to submit using the Kaggle API or submit via the Kaggle notebook interface.

### ⚠️ System Requirements for Running
- **Kaggle Environment**: This notebook must be run on Kaggle (due to GPU VRAM limits, standard 2GB GPUs cannot run this 30B model).
- **Internet**: Toggle the **Internet** setting to **On** in the right sidebar of the Kaggle editor.

## 🛠️ Step 1: Install & Set Up Dependencies

First, we make sure that the required utility libraries, PEFT, and model wrappers are available.

In [ ]:
import os
import site
import subprocess

# Setup paths for NVIDIA Cutlass DSL and utilities (provided in Kaggle environment)
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
if os.path.exists(cutlass_pkg_path):
    site.addsitedir(cutlass_pkg_path)
    print("NVIDIA Cutlass DSL package path loaded successfully.")
else:
    print("NVIDIA Cutlass package path not found. If running locally, please ignore.")

## 📥 Step 2: Download the Model & Initialize LoRA Adapter

We load the base model `NVIDIA Nemotron-3-Nano-30B` in `bfloat16` precision and initialize a LoRA adapter wrapper.

In [ ]:
import kagglehub
import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM

# Define model path and output location
print("Downloading / verifying baseline model...")
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32  # Maximum rank allowed is 32

print(f"Model downloaded to: {MODEL_PATH}")
print("Loading base model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)
print("Base model loaded successfully.")

# Setup PEFT LoRA Config matching Nemotron's architecture projection layers
print(f"Initializing LoRA configuration (Rank = {LORA_RANK})...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Wrap base model with adapter parameters
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 💾 Step 3: Save and Package the Adapter

We save the configuration and weight files to the working directory, and then zip them to create the submission package.

In [ ]:
print(f"Saving LoRA adapter files to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)

# Zip the contents at the root level of the working directory
zip_cmd = "zip -j submission.zip /kaggle/working/adapter_config.json /kaggle/working/adapter_model.safetensors"
print(f"Executing: {zip_cmd}")
subprocess.run(zip_cmd, shell=True, check=True)
print("submission.zip created successfully!")

## 📤 Step 4: Submit to Leaderboard

We can submit the file directly to Kaggle using the CLI tool. If you have your Kaggle credentials set up, you can uncomment and run the cell below.

In [ ]:
# !kaggle competitions submit -c nvidia-nemotron-model-reasoning-challenge -f submission.zip -m "Baseline adapter submission test"